In [4]:
import torch
from Bio import SeqIO
from BCBio import GFF
import random

# load the model
model_path = "../external/megaDNA_phage_145M.pt" # model name
device = 'cuda' if torch.cuda.is_available() else 'cpu'  # Device configuration

model = torch.load(model_path, map_location=torch.device(device), weights_only=False)
model.eval() 

/home/hananel/python_projects/megadna_embeddings2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MEGADNA(
  (start_tokens): ParameterList(
      (0): Parameter containing: [torch.float32 of size 512 (cuda:0)]
      (1): Parameter containing: [torch.float32 of size 256 (cuda:0)]
      (2): Parameter containing: [torch.float32 of size 196 (cuda:0)]
  )
  (token_embs): ModuleList(
    (0): Embedding(6, 196)
    (1): Sequential(
      (0): Embedding(6, 196)
      (1): Rearrange('... r d -> ... (r d)')
      (2): LayerNorm((3136,), eps=1e-05, elementwise_affine=True)
      (3): Linear(in_features=3136, out_features=256, bias=True)
      (4): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    )
    (2): Sequential(
      (0): Embedding(6, 196)
      (1): Rearrange('... r d -> ... (r d)')
      (2): LayerNorm((200704,), eps=1e-05, elementwise_affine=True)
      (3): Linear(in_features=200704, out_features=512, bias=True)
      (4): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    )
  )
  (transformers): ModuleList(
    (0): Transformer(
      (layers): ModuleList(
       

In [6]:
limit_info

{'gff_type': ['CDS']}

Please download the fasta file and gene annotation for lambda phage from https://www.ncbi.nlm.nih.gov/nuccore/NC_001416.1

In [8]:
# Read the FASTA file
fasta_file_path = "sequence (1).fasta"
seq_ids, sequences = [], []

with open(fasta_file_path, "r") as fasta_file:
    for record in SeqIO.parse(fasta_file, "fasta"):
        seq_ids.append(record.id)
        sequences.append(str(record.seq))

# Read the gene annotations
gff_file_path = "sequence (1).gff3"
limit_info = dict(gff_type=["CDS"])

start_position, end_position, strand_position = [], [], []

with open(gff_file_path) as in_handle:
    for rec in GFF.parse(in_handle, limit_info=limit_info):
        start_position.extend(feature.location.start for feature in rec.features)
        end_position.extend(feature.location.end for feature in rec.features)
        strand_position.extend(feature.location.strand for feature in rec.features)


In [9]:
nt = ['**', 'A', 'T', 'C', 'G', '#']  # Vocabulary
seq_id = 0  # Sequence ID

def encode_sequence(sequence, nt_vocab=nt):
    """Encode a DNA sequence to its numerical representation."""
    return [0] + [nt_vocab.index(nucleotide) if nucleotide in nt_vocab else 1 for nucleotide in sequence] + [5]

def get_loss_for_sequence(model, sequence, device):
    """Get model loss for a given sequence."""
    input_seq = torch.tensor(sequence).unsqueeze(0).to(device)
    with torch.no_grad():
        loss = model(input_seq, return_value='loss')
    return loss

# Get the model loss for the WT sequence
encoded_wt_sequence = encode_sequence(sequences[seq_id])
wt_loss = get_loss_for_sequence(model, encoded_wt_sequence, device)
print(wt_loss)

# Get the model loss for the mutants in the start codons
loss_start = []
random.seed(42)
for j, (start, end, strand) in enumerate(zip(start_position, end_position, strand_position)):
    encoded_mutant_sequence = encode_sequence(sequences[seq_id])
    
    # Mutate start codon positions based on strand orientation
    positions = range(start+1, start+4) if strand == 1 else range(end-2, end+1)
    for i in positions:
        encoded_mutant_sequence[i] = random.choice([1, 2, 3, 4])
    
    # Get model loss for mutated sequence
    mutant_loss = get_loss_for_sequence(model, encoded_mutant_sequence, device)
    loss_start.append(mutant_loss)

/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


tensor(1.2624, device='cuda:0')


In [12]:
print(wt_loss.item())
print([loss.item() for loss in loss_start])

1.2624249458312988
[1.2624698877334595, 1.2625478506088257, 1.2628728151321411, 1.262574315071106, 1.262556552886963, 1.2624444961547852, 1.262641429901123, 1.2625657320022583, 1.2625607252120972, 1.2627087831497192, 1.262637972831726, 1.2628353834152222, 1.2626185417175293, 1.2627332210540771, 1.2623614072799683, 1.2629013061523438, 1.2627074718475342, 1.262771725654602, 1.2624444961547852, 1.2627111673355103, 1.2627437114715576, 1.262546181678772, 1.2630360126495361, 1.262455940246582, 1.2624340057373047, 1.2627606391906738, 1.2625678777694702, 1.2624469995498657, 1.2625579833984375, 1.2624289989471436, 1.2624588012695312, 1.26261568069458, 1.2626149654388428, 1.2626038789749146, 1.2624554634094238, 1.2625689506530762, 1.2625128030776978, 1.2626967430114746, 1.2625982761383057, 1.2624472379684448, 1.2625318765640259, 1.262555480003357, 1.2626386880874634, 1.2625858783721924, 1.2624945640563965, 1.2624249458312988, 1.262420654296875, 1.2624516487121582, 1.2625519037246704, 1.262541890